# Dynamic Withdrawal Strategies - Daily Path Analysis

이 노트북은 Dynamic 전략의 일별 경로 데이터를 분석하고 Excel 검증을 수행합니다.

## Step 1: 라이브러리 및 데이터 로드

In [9]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


In [10]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")

✅ 벤치마크 데이터 로드 완료
   데이터 크기: (6521, 8)
   날짜 범위: 2001-01-03 ~ 2025-12-31


## Step 2: DataPreprocessor

In [11]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

preprocessor = DataPreprocessor(data, add_portfolios=True)
returns_df, month_starts = preprocessor.get_data()

print(f"✅ DataPreprocessor 완료")
print(f"   Returns DataFrame: {returns_df.shape}")
print(f"   Month Starts Series: {month_starts.shape}")

=== 데이터 전처리 시작 ===
데이터 기간: 2001-01-03 ~ 2025-12-31
총 거래일: 6,521일
벤치마크: 8개

일별 수익률 계산 중... ✅ (6,520개 수익률)

수익률 통계 (연율화):
         연평균수익률   연변동성
미국성장주     12.16  19.90
국내주식      14.40  28.11
미국국채       4.18  11.06
미국외국채      3.34  11.36
신흥국달러채권    7.45  10.59
국내중기채      3.86   2.37
국내장기채      6.16   8.13
금         13.13  19.50

포트폴리오 수익률 계산:
  추가할 포트폴리오: 6개
  ✅ Port_4.0%: 연수익률 4.92%, 연변동성 2.65%
  ✅ Port_5.0%: 연수익률 6.01%, 연변동성 3.81%
  ✅ Port_6.0%: 연수익률 7.10%, 연변동성 5.39%
  ✅ Port_7.0%: 연수익률 8.28%, 연변동성 7.09%
  ✅ Port_8.0%: 연수익률 9.57%, 연변동성 8.95%
  ✅ Port_9.0%: 연수익률 10.87%, 연변동성 10.89%

월초 거래일 식별 중... ✅ (300개월)
첫 10개 월초: [datetime.date(2001, 1, 4), datetime.date(2001, 2, 1), datetime.date(2001, 3, 1), datetime.date(2001, 4, 2), datetime.date(2001, 5, 1), datetime.date(2001, 6, 1), datetime.date(2001, 7, 2), datetime.date(2001, 8, 1), datetime.date(2001, 9, 3), datetime.date(2001, 10, 1)]
✅ 전처리 완료

✅ DataPreprocessor 완료
   Returns DataFrame: (6520, 14)
   Month Starts Series: (6520,)


## Step 3: Dynamic Simulator 생성

In [12]:
from dynamic_simulator import DynamicWithdrawalSimulator

simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

✅ DynamicWithdrawalSimulator 생성 완료
   총 날짜: 6520
   월초 개수: 300


## Step 4: 전략 파라미터 설정 (통합)

여기서 모든 전략의 파라미터를 한번에 설정합니다.

In [ ]:
# ============================================================
# 공통 파라미터
# ============================================================
start_date = '2007-10-01'  # 시작일 (금융위기 직전으로 설정 - Status 섞이도록)
test_portfolio = 'Port_5.0%'  # 포트폴리오
horizon_years = 10  # 시뮬레이션 기간 (년)
initial_wr = 0.08  # 초기 인출률 (8% - 높게 설정하여 Breach 유도)
v0 = 100.0  # 초기 NAV

# ============================================================
# Guardrails 전략 파라미터
# ============================================================
guardrails_params = {
    'guardrail_width': 0.20,         # ±20%
    'adjustment_pct': 0.0,           # 0 = cap 모드, > 0 = adjust 모드
    'return_threshold': 0.05         # ±5% (adjust 모드에서만 사용)
}

# ============================================================
# Fixed Rate 전략 파라미터
# ============================================================
# Fixed는 추가 파라미터 없음

# ============================================================
# 포트폴리오 정보 출력
# ============================================================
print(f"\n{'='*60}")
print(f"전략 파라미터 설정 완료")
print(f"{'='*60}")
print(f"\n공통 설정:")
print(f"  시작일: {start_date}")
print(f"  포트폴리오: {test_portfolio}")
print(f"  시뮬레이션 기간: {horizon_years}년")
print(f"  초기 인출률: {initial_wr*100:.1f}%")
print(f"  초기 NAV: {v0:.0f}")

portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    print(f"\n포트폴리오 구성:")
    print(f"  목표 수익률: {portfolio_config['target_return']:.2f}%")
    print(f"  목표 변동성: {portfolio_config['target_risk']:.2f}%")
    print(f"\n  자산 구성:")
    total_weight = 0.0
    for asset_kor, weight_pct in portfolio_config['weights'].items():
        print(f"    {asset_kor:15s}: {weight_pct:6.2f}%")
        total_weight += weight_pct
    print(f"    {'-'*30}")
    print(f"    {'합계':15s}: {total_weight:6.2f}%")

print(f"\nGuardrails 파라미터:")
print(f"  상한: {initial_wr*(1+guardrails_params['guardrail_width'])*100:.1f}%")
print(f"  하한: {initial_wr*(1-guardrails_params['guardrail_width'])*100:.1f}%")
print(f"  조정 모드: {'adjust' if guardrails_params['adjustment_pct'] > 0 else 'cap'}")
if guardrails_params['adjustment_pct'] > 0:
    print(f"  조정 비율: ±{guardrails_params['adjustment_pct']*100:.0f}%")
    print(f"  수익률 임계값: ±{guardrails_params['return_threshold']*100:.0f}%")

## Step 5: Guardrails 일별 경로 조회

In [ ]:
# ============================================================
# Guardrails 전략 파라미터 출력
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrails 전략 - 인출액 계산 로직")
print(f"{'='*60}")
guardrail_width = guardrails_params['guardrail_width']
adjustment_pct = guardrails_params['adjustment_pct']
return_threshold = guardrails_params['return_threshold']

# 모드 자동 판정
mode = 'adjust' if adjustment_pct > 0 else 'cap'

if mode == 'cap':
    print(f"\n  📍 Cap 모드 (NAV 기준 guardrail)")
    print(f"     상한 위반 시 (current_wr > {initial_wr*(1+guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 현재 NAV × {initial_wr*(1+guardrail_width)*100:.1f}% / 12")
    print(f"     ➜ 즉시 상한으로 재조정 (NAV 기준)")
    print(f"\n     하한 위반 시 (current_wr < {initial_wr*(1-guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 현재 NAV × {initial_wr*(1-guardrail_width)*100:.1f}% / 12")
    print(f"     ➜ 즉시 하한으로 재조정 (NAV 기준)")
    print(f"\n     정상 범위 ({initial_wr*(1-guardrail_width)*100:.1f}% ≤ current_wr ≤ {initial_wr*(1+guardrail_width)*100:.1f}%):")
    print(f"     ➜ 기본 인출액 유지")
else:
    print(f"\n  📍 Adjust 모드 (전월 수익률 기준)")
    print(f"     수익률 악화 시 (monthly_return < -{return_threshold*100:.0f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 - {adjustment_pct*100:.0f}%)")
    print(f"     ➜ {adjustment_pct*100:.0f}% 감액")
    print(f"\n     수익률 개선 시 (monthly_return > +{return_threshold*100:.0f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 + {adjustment_pct*100:.0f}%)")
    print(f"     ➜ {adjustment_pct*100:.0f}% 증액")
    print(f"\n     정상 범위 (-{return_threshold*100:.0f}% ≤ monthly_return ≤ +{return_threshold*100:.0f}%):")
    print(f"     ➜ 기본 인출액 유지")

# ============================================================
# 일별 경로 조회
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

guardrails_daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='guardrails',
    horizon_years=horizon_years,
    initial_wr=initial_wr,
    guardrail_width=guardrails_params['guardrail_width'],
    guardrail_adjustment_pct=guardrails_params['adjustment_pct'],
    guardrail_return_threshold=guardrails_params['return_threshold'],
    v0=v0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {guardrails_daily_path_df.shape}")
print(f"  날짜 범위: {guardrails_daily_path_df['Date'].iloc[0].date()} ~ {guardrails_daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(guardrails_daily_path_df)}")

price_cols = [c for c in guardrails_daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in guardrails_daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# ============================================================
# Guardrail Status 분석
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail Status 분포")
print(f"{'='*60}")

status_counts = guardrails_daily_path_df['Guardrail_Status'].value_counts()
print(f"\n전체 {len(guardrails_daily_path_df)}일 중:")
for status, count in status_counts.items():
    pct = count / len(guardrails_daily_path_df) * 100
    print(f"  {status:15s}: {count:4d}일 ({pct:5.2f}%)")

month_starts_df = guardrails_daily_path_df[guardrails_daily_path_df['Is_Month_Start'] == True]
if len(month_starts_df) > 0:
    month_status_counts = month_starts_df['Guardrail_Status'].value_counts()
    print(f"\n월초 {len(month_starts_df)}개월 중:")
    for status, count in month_status_counts.items():
        pct = count / len(month_starts_df) * 100
        print(f"  {status:15s}: {count:4d}개월 ({pct:5.2f}%)")

# ============================================================
# 월초 인출액 및 Guardrail 위반 사례 확인
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail 위반 사례")
print(f"{'='*60}")
if len(guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)]) > 0:
    break_threshold = guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)][
        ['Date', 'Total_NAV', 'Withdrawal_Amount', 'Current_WR', 'Guardrail_Status', 'Year_Month']
    ].head(10)
    display(len(guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)]))
    display(break_threshold)    
else:
    print("위반 사례 없음")

# Excel 내보내기
guardrails_daily_path_df.to_excel('guardrails_path_details.xlsx', index=False)
print(f"\n✅ Excel 파일 저장 완료: guardrails_path_details.xlsx")

## Step 6: Fixed Rate 일별 경로 조회

In [ ]:
# ============================================================
# Fixed Rate 전략 파라미터 출력
# ============================================================
print(f"\n{'='*60}")
print(f"Fixed Rate 전략 - 인출액 계산 로직")
print(f"{'='*60}")
print(f"\n  📍 완전 고정 인출:")
print(f"     ➜ 첫 달: 초기 NAV × {initial_wr*100:.1f}% / 12")
print(f"     ➜ 이후: 동일 금액 영구 고정")
print(f"     ➜ Guardrail 없음, 재계산 없음")

# ============================================================
# 일별 경로 조회
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

fixed_daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='fixed',
    horizon_years=horizon_years,
    initial_wr=initial_wr,
    v0=v0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {fixed_daily_path_df.shape}")
print(f"  날짜 범위: {fixed_daily_path_df['Date'].iloc[0].date()} ~ {fixed_daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(fixed_daily_path_df)}")

price_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# ============================================================
# Guardrail Status 분석
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail Status 분포")
print(f"{'='*60}")

status_counts = fixed_daily_path_df['Guardrail_Status'].value_counts()
print(f"\n전체 {len(fixed_daily_path_df)}일 중:")
for status, count in status_counts.items():
    pct = count / len(fixed_daily_path_df) * 100
    print(f"  {status:15s}: {count:4d}일 ({pct:5.2f}%)")

month_starts_df = fixed_daily_path_df[fixed_daily_path_df['Is_Month_Start'] == True]
if len(month_starts_df) > 0:
    month_status_counts = month_starts_df['Guardrail_Status'].value_counts()
    print(f"\n월초 {len(month_starts_df)}개월 중:")
    for status, count in month_status_counts.items():
        pct = count / len(month_starts_df) * 100
        print(f"  {status:15s}: {count:4d}개월 ({pct:5.2f}%)")

# ============================================================
# 월초 인출액 및 Guardrail 위반 사례 확인
# ============================================================
print(f"\n{'='*60}")
print(f"월초 인출액 및 Guardrail Status")
print(f"{'='*60}")
if len(fixed_daily_path_df[fixed_daily_path_df['Guardrail_Status'] != 'Normal']) > 0:
    break_threshold = fixed_daily_path_df[fixed_daily_path_df['Guardrail_Status'] != 'Normal'][
        ['Date', 'Total_NAV', 'Withdrawal_Amount', 'Current_WR', 'Guardrail_Status', 'Year_Month']
    ]
    display(break_threshold)
else:
    print("위반 사례 없음")

# Excel 내보내기
fixed_daily_path_df.to_excel('fixed_rate_path_details.xlsx', index=False)
print(f"\n✅ Excel 파일 저장 완료: fixed_rate_path_details.xlsx")